In [0]:
df_hosp_a = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/A_Hospedagens.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df_hosp_b = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/B_Hospedagens.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

In [0]:
display(df_hosp_a)
display(df_hosp_b)

Voucher,Hospede,Check-in,Check-out,Centro Custo,Cidade,Total Hospedagem,Status
HVN-2001,Henrique Lima,2026-05-18,2026-05-23,COSAD,Recife,"R$ 551,76",Confirmada
HVN-2002,Vinicius Reis,2026-09-23,2026-09-25,GPES,Brasília,"R$ 1.600,70",Confirmada
HVN-2003,Renata Cardoso,2026-08-16,2026-08-21,GPES,Salvador,"R$ 2.262,88",Confirmada
HVN-2004,Isabela Rocha,2026-06-22,2026-06-25,GTI,Recife,"R$ 574,84",Confirmada
HVN-2005,Karen Duarte,2026-10-13,2026-10-16,GJUR,São Paulo,"R$ 319,56",Confirmada
HVN-2006,Carla Ribeiro,2026-10-07,2026-10-12,COSAD,Belo Horizonte,"R$ 1.362,36",Confirmada
HVN-2007,Henrique Lima,2026-06-03,2026-06-06,COSAD,Salvador,"R$ 898,47",Confirmada
HVN-2008,João Monteiro,2026-08-21,2026-08-26,GCOM,Brasília,"R$ 1.702,51",Confirmada
HVN-2009,Diego Freitas,2026-09-19,2026-09-22,GCOM,Rio de Janeiro,"R$ 855,38",Confirmada
HVN-2010,Isabela Rocha,2026-05-28,2026-06-02,GTI,Curitiba,"R$ 1.570,14",Confirmada


Booking,Nome Passageiro,Entrada,Saida,Unidade,Municipio,Valor NF,Situacao
HAT-8001,Isabela Rocha,2026-07-25,2026-07-27,Comunicação,Rio de Janeiro,"R$ 1.856,58",OK
HAT-8002,Ana Martins,2026-09-19,2026-09-23,Gestão de Pessoas,São Paulo,"R$ 458,01",OK
HAT-8003,Bruno Nogueira,2026-10-24,2026-10-26,Gestão de Pessoas,Salvador,"R$ 1.523,27",OK
HAT-8004,Bruno Nogueira,2026-06-07,2026-06-11,Ger. Tecnologia,Curitiba,"R$ 1.065,77",OK
HAT-8005,Henrique Lima,2026-05-11,2026-05-14,Ger. Tecnologia,São Paulo,"R$ 1.042,73",OK
HAT-8006,Eduarda Melo,2026-09-21,2026-09-25,Gestão de Pessoas,Belo Horizonte,"R$ 2.115,67",OK
HAT-8007,Felipe Castro,2026-06-15,2026-06-19,Coord. Suprimentos,Salvador,"R$ 1.943,01",OK
HAT-8008,Felipe Castro,2026-08-14,2026-08-16,Financeiro,São Paulo,"R$ 1.103,34",OK
HAT-8009,Gabriela Torres,2026-05-10,2026-05-14,Ger. Tecnologia,Goiânia,"R$ 1.034,28",OK
HAT-8010,Gabriela Torres,2026-06-28,2026-06-30,Coord. Suprimentos,Recife,"R$ 989,35",OK


In [0]:
df_hosp_a.printSchema()
df_hosp_b.printSchema()

df_hosp_a.columns
df_hosp_b.columns

root
 |-- Voucher: string (nullable = true)
 |-- Hospede: string (nullable = true)
 |-- Check-in: date (nullable = true)
 |-- Check-out: date (nullable = true)
 |-- Centro Custo: string (nullable = true)
 |-- Cidade: string (nullable = true)
 |-- Total Hospedagem: string (nullable = true)
 |-- Status: string (nullable = true)

root
 |-- Booking: string (nullable = true)
 |-- Nome Passageiro: string (nullable = true)
 |-- Entrada: date (nullable = true)
 |-- Saida: date (nullable = true)
 |-- Unidade: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Valor NF: string (nullable = true)
 |-- Situacao: string (nullable = true)



['Booking',
 'Nome Passageiro',
 'Entrada',
 'Saida',
 'Unidade',
 'Municipio',
 'Valor NF',
 'Situacao']

In [0]:
from pyspark.sql.functions import col, lit

In [0]:
df_hosp_a_padrao = df_hosp_a.select(
    col("Voucher").alias("id_hospedagem"),
    col("Hospede").alias("viajante"),
    col("Check-in").alias("data_checkin"),
    col("Check-out").alias("data_checkout"),
    col("Centro Custo").alias("centro_custo_origem"),
    col("Cidade").alias("cidade"),
    col("Total Hospedagem").alias("valor"),
    col("Status").alias("situacao")
).withColumn(
    "agencia",
    lit("ViaNorte")
)

In [0]:
df_hosp_b_padrao = df_hosp_b.select(
    col("Booking").alias("id_hospedagem"),
    col("Nome Passageiro").alias("viajante"),
    col("Entrada").alias("data_checkin"),
    col("Saida").alias("data_checkout"),
    col("Unidade").alias("centro_custo_origem"),
    col("Municipio").alias("cidade"),
    col("Valor NF").alias("valor"),
    col("Situacao").alias("situacao")
).withColumn(
    "agencia",
    lit("Atlas Corporate")
)

In [0]:
df_hosp_a_padrao.columns == df_hosp_b_padrao.columns

True

In [0]:
df_hosp_a_padrao.count()
df_hosp_b_padrao.count()

199

In [0]:
df_hosp_a_padrao = df_hosp_a_padrao.filter(
    col("id_hospedagem").isNotNull()
)

df_hosp_b_padrao = df_hosp_b_padrao.filter(
    col("id_hospedagem").isNotNull()
)

In [0]:
print(df_hosp_a_padrao.count())
print(df_hosp_b_padrao.count())

12
12


In [0]:
df_hospedagens = df_hosp_a_padrao.unionByName(
    df_hosp_b_padrao
)

In [0]:
df_hospedagens.count()

24

In [0]:
display(df_hospedagens)

id_hospedagem,viajante,data_checkin,data_checkout,centro_custo_origem,cidade,valor,situacao,agencia
HVN-2001,Henrique Lima,2026-05-18,2026-05-23,COSAD,Recife,"R$ 551,76",Confirmada,ViaNorte
HVN-2002,Vinicius Reis,2026-09-23,2026-09-25,GPES,Brasília,"R$ 1.600,70",Confirmada,ViaNorte
HVN-2003,Renata Cardoso,2026-08-16,2026-08-21,GPES,Salvador,"R$ 2.262,88",Confirmada,ViaNorte
HVN-2004,Isabela Rocha,2026-06-22,2026-06-25,GTI,Recife,"R$ 574,84",Confirmada,ViaNorte
HVN-2005,Karen Duarte,2026-10-13,2026-10-16,GJUR,São Paulo,"R$ 319,56",Confirmada,ViaNorte
HVN-2006,Carla Ribeiro,2026-10-07,2026-10-12,COSAD,Belo Horizonte,"R$ 1.362,36",Confirmada,ViaNorte
HVN-2007,Henrique Lima,2026-06-03,2026-06-06,COSAD,Salvador,"R$ 898,47",Confirmada,ViaNorte
HVN-2008,João Monteiro,2026-08-21,2026-08-26,GCOM,Brasília,"R$ 1.702,51",Confirmada,ViaNorte
HVN-2009,Diego Freitas,2026-09-19,2026-09-22,GCOM,Rio de Janeiro,"R$ 855,38",Confirmada,ViaNorte
HVN-2010,Isabela Rocha,2026-05-28,2026-06-02,GTI,Curitiba,"R$ 1.570,14",Confirmada,ViaNorte


In [0]:
from pyspark.sql.functions import regexp_replace

In [0]:
df_hospedagens = df_hospedagens.withColumn(
    "valor_numerico",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("valor"),
                "R\\$ ",
                ""
            ),
            "\\.",
            ""
        ),
        ",",
        "."
    ).cast("decimal(18,2)")
)

In [0]:
df_hospedagens.select(
    "valor",
    "valor_numerico"
).show(20, truncate=False)

+-----------+--------------+
|valor      |valor_numerico|
+-----------+--------------+
|R$ 551,76  |551.76        |
|R$ 1.600,70|1600.70       |
|R$ 2.262,88|2262.88       |
|R$ 574,84  |574.84        |
|R$ 319,56  |319.56        |
|R$ 1.362,36|1362.36       |
|R$ 898,47  |898.47        |
|R$ 1.702,51|1702.51       |
|R$ 855,38  |855.38        |
|R$ 1.570,14|1570.14       |
|R$ 827,35  |827.35        |
|R$ 392,58  |392.58        |
|R$ 1.856,58|1856.58       |
|R$ 458,01  |458.01        |
|R$ 1.523,27|1523.27       |
|R$ 1.065,77|1065.77       |
|R$ 1.042,73|1042.73       |
|R$ 2.115,67|2115.67       |
|R$ 1.943,01|1943.01       |
|R$ 1.103,34|1103.34       |
+-----------+--------------+
only showing top 20 rows


In [0]:
df_hospedagens.printSchema()

root
 |-- id_hospedagem: string (nullable = true)
 |-- viajante: string (nullable = true)
 |-- data_checkin: date (nullable = true)
 |-- data_checkout: date (nullable = true)
 |-- centro_custo_origem: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- situacao: string (nullable = true)
 |-- agencia: string (nullable = false)
 |-- valor_numerico: decimal(18,2) (nullable = true)



In [0]:
from pyspark.sql.functions import datediff

In [0]:
df_hospedagens = df_hospedagens.withColumn(
    "qtd_noites",
    datediff(
        col("data_checkout"),
        col("data_checkin")
    )
)

In [0]:
df_hospedagens.select(
    "viajante",
    "data_checkin",
    "data_checkout",
    "qtd_noites"
).show(20, truncate=False)

+--------------+------------+-------------+----------+
|viajante      |data_checkin|data_checkout|qtd_noites|
+--------------+------------+-------------+----------+
|Henrique Lima |2026-05-18  |2026-05-23   |5         |
|Vinicius Reis |2026-09-23  |2026-09-25   |2         |
|Renata Cardoso|2026-08-16  |2026-08-21   |5         |
|Isabela Rocha |2026-06-22  |2026-06-25   |3         |
|Karen Duarte  |2026-10-13  |2026-10-16   |3         |
|Carla Ribeiro |2026-10-07  |2026-10-12   |5         |
|Henrique Lima |2026-06-03  |2026-06-06   |3         |
|João Monteiro |2026-08-21  |2026-08-26   |5         |
|Diego Freitas |2026-09-19  |2026-09-22   |3         |
|Isabela Rocha |2026-05-28  |2026-06-02   |5         |
|Isabela Rocha |2026-10-31  |2026-11-03   |3         |
|Nicolas Souza |2026-05-14  |2026-05-15   |1         |
|Isabela Rocha |2026-07-25  |2026-07-27   |2         |
|Ana Martins   |2026-09-19  |2026-09-23   |4         |
|Bruno Nogueira|2026-10-24  |2026-10-26   |2         |
|Bruno Nog

In [0]:
df_hospedagens = df_hospedagens.withColumn(
    "tipo_despesa",
    lit("Hospedagem")
)

In [0]:
df_hospedagens.select(
    "id_hospedagem",
    "tipo_despesa"
).show(10)

+-------------+------------+
|id_hospedagem|tipo_despesa|
+-------------+------------+
|     HVN-2001|  Hospedagem|
|     HVN-2002|  Hospedagem|
|     HVN-2003|  Hospedagem|
|     HVN-2004|  Hospedagem|
|     HVN-2005|  Hospedagem|
|     HVN-2006|  Hospedagem|
|     HVN-2007|  Hospedagem|
|     HVN-2008|  Hospedagem|
|     HVN-2009|  Hospedagem|
|     HVN-2010|  Hospedagem|
+-------------+------------+
only showing top 10 rows


In [0]:
df_map = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/Centro_Custo_Map.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

In [0]:
df_hospedagens = (
    df_hospedagens.alias("h")
    .join(
        df_map.alias("m"),
        (
            (col("h.agencia") == col("m.Agencia")) &
            (col("h.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "h.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_hospedagens.select(
    "agencia",
    "centro_custo_origem",
    "centro_custo",
    "diretoria"
).show(30, truncate=False)

+---------------+-------------------+------------+---------+
|agencia        |centro_custo_origem|centro_custo|diretoria|
+---------------+-------------------+------------+---------+
|ViaNorte       |COSAD              |COSAD       |DIRADM   |
|ViaNorte       |GPES               |GPES        |DIRPES   |
|ViaNorte       |GPES               |GPES        |DIRPES   |
|ViaNorte       |GTI                |GTI         |DIRTEC   |
|ViaNorte       |GJUR               |GJUR        |DIRJUR   |
|ViaNorte       |COSAD              |COSAD       |DIRADM   |
|ViaNorte       |COSAD              |COSAD       |DIRADM   |
|ViaNorte       |GCOM               |GCOM        |DIRCOM   |
|ViaNorte       |GCOM               |GCOM        |DIRCOM   |
|ViaNorte       |GTI                |GTI         |DIRTEC   |
|ViaNorte       |GTI                |GTI         |DIRTEC   |
|ViaNorte       |GCOM               |GCOM        |DIRCOM   |
|Atlas Corporate|Comunicação        |GCOM        |DIRCOM   |
|Atlas Corporate|Gestão 

In [0]:
df_hospedagens.filter(
    col("centro_custo").isNull()
).count()

0

In [0]:
df_hospedagens.count()

24

In [0]:
from pyspark.sql.functions import month, when

In [0]:
df_hospedagens = df_hospedagens.withColumn(
    "trimestre",
    when(
        month(col("data_checkin")).between(1, 3),
        "Primeiro Trimestre"
    )
    .when(
        month(col("data_checkin")).between(4, 6),
        "Segundo Trimestre"
    )
    .when(
        month(col("data_checkin")).between(7, 9),
        "Terceiro Trimestre"
    )
    .otherwise(
        "Quarto Trimestre"
    )
)

In [0]:
df_hospedagens.groupBy("trimestre").count().show()

+------------------+-----+
|         trimestre|count|
+------------------+-----+
|  Quarto Trimestre|    4|
| Segundo Trimestre|   11|
|Terceiro Trimestre|    9|
+------------------+-----+



In [0]:
df_hospedagens.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.silver.hospedagens")

In [0]:
df_hosp_teste = spark.table(
    "viagens_teste.silver.hospedagens"
)

display(df_hosp_teste)

id_hospedagem,viajante,data_checkin,data_checkout,centro_custo_origem,cidade,valor,situacao,agencia,valor_numerico,qtd_noites,tipo_despesa,centro_custo,diretoria,trimestre
HAT-8001,Isabela Rocha,2026-07-25,2026-07-27,Comunicação,Rio de Janeiro,"R$ 1.856,58",OK,Atlas Corporate,1856.58,2,Hospedagem,GCOM,DIRCOM,Terceiro Trimestre
HAT-8002,Ana Martins,2026-09-19,2026-09-23,Gestão de Pessoas,São Paulo,"R$ 458,01",OK,Atlas Corporate,458.01,4,Hospedagem,GPES,DIRPES,Terceiro Trimestre
HAT-8003,Bruno Nogueira,2026-10-24,2026-10-26,Gestão de Pessoas,Salvador,"R$ 1.523,27",OK,Atlas Corporate,1523.27,2,Hospedagem,GPES,DIRPES,Quarto Trimestre
HAT-8004,Bruno Nogueira,2026-06-07,2026-06-11,Ger. Tecnologia,Curitiba,"R$ 1.065,77",OK,Atlas Corporate,1065.77,4,Hospedagem,GTI,DIRTEC,Segundo Trimestre
HAT-8005,Henrique Lima,2026-05-11,2026-05-14,Ger. Tecnologia,São Paulo,"R$ 1.042,73",OK,Atlas Corporate,1042.73,3,Hospedagem,GTI,DIRTEC,Segundo Trimestre
HAT-8006,Eduarda Melo,2026-09-21,2026-09-25,Gestão de Pessoas,Belo Horizonte,"R$ 2.115,67",OK,Atlas Corporate,2115.67,4,Hospedagem,GPES,DIRPES,Terceiro Trimestre
HAT-8007,Felipe Castro,2026-06-15,2026-06-19,Coord. Suprimentos,Salvador,"R$ 1.943,01",OK,Atlas Corporate,1943.01,4,Hospedagem,COSAD,DIRADM,Segundo Trimestre
HAT-8008,Felipe Castro,2026-08-14,2026-08-16,Financeiro,São Paulo,"R$ 1.103,34",OK,Atlas Corporate,1103.34,2,Hospedagem,GFIN,DIRFIN,Terceiro Trimestre
HAT-8009,Gabriela Torres,2026-05-10,2026-05-14,Ger. Tecnologia,Goiânia,"R$ 1.034,28",OK,Atlas Corporate,1034.28,4,Hospedagem,GTI,DIRTEC,Segundo Trimestre
HAT-8010,Gabriela Torres,2026-06-28,2026-06-30,Coord. Suprimentos,Recife,"R$ 989,35",OK,Atlas Corporate,989.35,2,Hospedagem,COSAD,DIRADM,Segundo Trimestre
